In [30]:
import avalanche as av
import networkx as nx
import numpy as np
import random
import time
import gudhi
import matplotlib.pyplot as plt

# Table 3

## Code for computing the homotopy type of the examples in Table 3 of the manuscript using elementary collapses

In [31]:
import gudhi
from itertools import chain, combinations


def powerset(iterable):
    s = list(iterable)
    return chain.from_iterable(
        combinations(s, r) for r in range(len(s)+1)
    )


def all_simplices(st):
    return [tuple(sorted(s[0])) for s in st.get_simplices()]


def maximal_simplices(st):

    simplices = all_simplices(st)

    maximal = []

    for s in simplices:

        is_maximal = True

        for t in simplices:
            if set(s) < set(t):
                is_maximal = False
                break

        if is_maximal:
            maximal.append(s)

    return maximal


def free_pairs(st):
    """
    Return all elementary collapse pairs (sigma, tau)
    where sigma is a free face of tau.
    """

    facets = maximal_simplices(st)

    result = []

    for tau in facets:

        tau_set = set(tau)

        for sigma in powerset(tau):

            sigma = tuple(sorted(sigma))

            if sigma == tau:
                continue

            sigma_set = set(sigma)

            count = 0

            for G in facets:
                if sigma_set.issubset(G):
                    count += 1

            if count == 1:
                result.append((sigma, tau))

    return result


def elementary_collapse(st, sigma, tau):
    """
    Correct elementary collapse:
    remove all simplices eta with

        sigma ⊆ eta ⊆ tau
    """

    sigma = set(sigma)
    tau = set(tau)

    simplices = all_simplices(st)

    new_st = gudhi.SimplexTree()

    for eta in simplices:

        eta_set = set(eta)

        # remove iff sigma ⊆ eta ⊆ tau
        if sigma.issubset(eta_set) and eta_set.issubset(tau):
            continue

        new_st.insert(eta)

    return new_st


def greedy_collapse(st, verbose=False):

    step = 0

    while True:

        pairs = free_pairs(st)

        if not pairs:
            break

        sigma, tau = pairs[0]

        if verbose:
            print(f"Step {step}: collapse {sigma} inside {tau}")

        st = elementary_collapse(st, sigma, tau)

        step += 1

    return st

In [32]:
# The undirected graph on 6 vertices, with the sink connected to the first vertex
P6_undirected = nx.DiGraph()
P6_undirected.add_edges_from([(i,i+1) for i in range(5)]+[(i+1,i) for i in range(5)])
sinks = [0]
uP6_M = nx.adjacency_matrix(P6_undirected).todense()

## 1) Configuration (1, 7, 0, 0, 2, 0)

In [33]:
c1 = (1, 7, 0, 0, 2, 0)
A = av.avalanche_complex(uP6_M,c1,sinks=sinks)
A_collapse=greedy_collapse(A,verbose=True)
A_collapse.persistence(persistence_dim_max=True)
print("Collapsed Betti numbers : ",A_collapse.betti_numbers(), " :: Original Betti_numbers",A.betti_numbers())
maximal_simplices(A_collapse)

Step 0: collapse (0, 1) inside (0, 1, 2, 3, 4, 6)
Step 1: collapse (0, 2) inside (0, 2, 3, 4, 6)
Step 2: collapse (0, 3) inside (0, 3, 4, 6)
Step 3: collapse (0, 6) inside (0, 4, 6)
Step 4: collapse (0,) inside (0, 4, 7)
Step 5: collapse (1, 2) inside (1, 2, 3, 4, 6)
Step 6: collapse (1, 4) inside (1, 3, 4, 6)
Step 7: collapse (1, 7) inside (1, 3, 5, 7)
Step 8: collapse (2, 3) inside (2, 3, 4, 6)
Step 9: collapse (2, 5) inside (2, 4, 5, 7)
Step 10: collapse (2, 6) inside (2, 4, 6)
Step 11: collapse (2,) inside (2, 4, 7)
Step 12: collapse (3, 4) inside (3, 4, 6)
Step 13: collapse (3, 7) inside (3, 5, 7)
Step 14: collapse (7,) inside (4, 5, 7)
Collapsed Betti numbers :  [1, 1, 1]  :: Original Betti_numbers [1, 1, 1, 0, 0, 0]


[(1, 3, 5), (1, 3, 6), (1, 5, 6), (3, 5, 6), (4, 5), (4, 6)]

### This is the hollow tetrahedra on vertices (1,3,5,6) and two edges connects vertex 4 to the tetraheda, thus we have S_1 v S_2


## 2) Configuration (4, 5, 0, 2, 3, 0)

In [34]:
c2 = (4, 5, 0, 2, 3, 0)
B = av.avalanche_complex(uP6_M,c2,sinks=sinks)
B_collapse=greedy_collapse(B,verbose=True)
B_collapse.persistence(persistence_dim_max=True)
print("Collapsed Betti numbers : ",B_collapse.betti_numbers(), " :: Original Betti_numbers",B.betti_numbers())
maximal_simplices(B_collapse)

Step 0: collapse (1, 2, 4) inside (0, 1, 2, 3, 4, 6)
Step 1: collapse (0, 1, 2, 5) inside (0, 1, 2, 3, 5)
Step 2: collapse (0, 1, 2) inside (0, 1, 2, 3, 6)
Step 3: collapse (0, 1, 3, 4) inside (0, 1, 3, 4, 6)
Step 4: collapse (0, 1, 4) inside (0, 1, 4, 5, 6)
Step 5: collapse (0, 2) inside (0, 2, 3, 4, 5, 6)
Step 6: collapse (0, 4) inside (0, 3, 4, 5, 6)
Step 7: collapse (1, 2) inside (1, 2, 3, 5, 6)
Step 8: collapse (2,) inside (2, 3, 4, 5, 6)
Collapsed Betti numbers :  [1, 0, 0, 2]  :: Original Betti_numbers [1, 0, 0, 2, 0, 0]


[(0, 1, 3, 5),
 (0, 1, 3, 6),
 (0, 1, 5, 6),
 (0, 3, 5, 6),
 (1, 3, 4, 5),
 (1, 3, 4, 6),
 (1, 3, 5, 6),
 (1, 4, 5, 6),
 (3, 4, 5, 6)]

### This is two spheres (0,1,3,5,6) and (1,3,4,5,6) glued along the face (1,3,5,6), thus we have S_3 v S_3

## 3) Configuration (7, 1, 2, 2, 2, 1)

In [35]:
c3 = (7, 1, 2, 2, 2, 1)
C = av.avalanche_complex(uP6_M,c3,sinks=sinks)
maximal_simplices(C)

[(0, 1, 2, 3, 4),
 (0, 1, 2, 3, 5),
 (0, 1, 2, 4, 5),
 (0, 1, 3, 4, 5),
 (0, 2, 3, 4, 5),
 (1, 2, 3, 4, 5)]

### These form a sphere on the vertices (0,1,2,3,4,5), thus we have S_4